# Glosario Interactivo — Conceptos Introductorios de ML

**Módulo 1 — Aprendizaje Supervisado** | DSRP Machine Learning Engineering

---

> 📖 Este notebook es un **glosario de referencia**. Cada concepto incluye una definición concisa y un bloque de código que lo ilustra visualmente.  
> No es necesario ejecutarlo de principio a fin — úsalo como consulta cuando encuentres un término desconocido en los otros notebooks.

### Índice
1. [Residuos](#1)
2. [MCO — Mínimos Cuadrados Ordinarios](#2)
3. [MLE — Máxima Verosimilitud](#3)
4. [Distribuciones de probabilidad](#4)
5. [Supuestos estadísticos](#5)
6. [Sesgo y Varianza (Bias-Variance Tradeoff)](#6)
7. [Overfitting y Underfitting](#7)
8. [Términos de clasificación (TP, FP, TN, FN)](#8)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

---
<a id='1'></a>
## 1. Residuos

**Definición:** El **residuo** de una observación es la diferencia entre el valor real $y_i$ y el valor predicho $\hat{y}_i$ por el modelo:

$$e_i = y_i - \hat{y}_i$$

Los residuos son la "huella" de los errores del modelo. Su análisis permite detectar:
- Si el modelo **subestima o sobreestima** sistemáticamente en alguna zona.
- Si los errores son **aleatorios** (buen modelo) o tienen algún patrón (modelo incompleto).

| Residuo | Significado |
|---------|-------------|
| $e_i > 0$ | El modelo predijo **menos** de lo real |
| $e_i < 0$ | El modelo predijo **más** de lo real |
| $e_i = 0$ | Predicción perfecta para esa observación |

In [ ]:
np.random.seed(0)
x = np.linspace(0, 10, 15)
y_real = 2 * x + 1 + np.random.normal(0, 2, len(x))  # datos reales con ruido
y_pred = 2 * x + 1                                    # línea de regresión (modelo)
residuos = y_real - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Izquierda: residuos como segmentos verticales ---
axes[0].plot(x, y_pred, 'b-', lw=2, label='Modelo ($\hat{y}$)')
axes[0].scatter(x, y_real, color='tomato', zorder=5, label='Datos reales ($y$)')
for xi, yr, yp in zip(x, y_real, y_pred):
    # Cada segmento vertical = residuo de ese punto
    color = 'green' if yr > yp else 'red'
    axes[0].plot([xi, xi], [yp, yr], color=color, lw=1.5, alpha=0.7)
axes[0].set_title('Residuos = distancia vertical entre dato real y modelo')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
pos_patch = mpatches.Patch(color='green', label='Residuo > 0 (subestimó)')
neg_patch = mpatches.Patch(color='red',   label='Residuo < 0 (sobreestimó)')
axes[0].legend(handles=[pos_patch, neg_patch], fontsize=8)
axes[0].grid(True, alpha=0.3)

# --- Derecha: distribución de residuos (queremos campana centrada en 0) ---
axes[1].hist(residuos, bins=8, edgecolor='white', color='steelblue', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', label='Residuo = 0')
axes[1].axvline(residuos.mean(), color='orange', ls='--', label=f'Media = {residuos.mean():.2f}')
axes[1].set_title('Distribución de residuos\n(ideal: campana centrada en 0)')
axes[1].set_xlabel('Residuo ($e_i$)'); axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

---
<a id='2'></a>
## 2. MCO — Mínimos Cuadrados Ordinarios (OLS)

**Definición:** MCO es el método que usa la **regresión lineal** para encontrar los coeficientes $\beta$ que minimizan la **suma de los residuos al cuadrado**:

$$\text{SCR} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

Se elevan al cuadrado para que residuos positivos y negativos no se cancelen, y para penalizar más los errores grandes.

La solución exacta en forma matricial es: $\hat{\beta} = (X^\top X)^{-1} X^\top y$

**¿Por qué "ordinarios"?** Existen variantes más complejas:
| Método | Cuándo se usa |
|--------|---------------|
| **MCO (OLS)** | Caso base — varianza constante, sin correlaciones entre errores |
| **GLS** (Mínimos Cuadrados Generalizados) | Cuando los errores tienen estructura de correlación conocida |
| **WLS** (Mínimos Cuadrados Ponderados) | Cuando algunas observaciones son más confiables que otras |
| **Ridge / Lasso** | MCO con regularización — evita sobreajuste con muchos predictores |

In [ ]:
np.random.seed(1)
x = np.linspace(0, 5, 20)
y = 1.5 * x + 2 + np.random.normal(0, 1, len(x))

# Probamos diferentes pendientes y calculamos la SCR (suma de cuadrados de residuos)
# MCO encuentra la pendiente que hace mínima esa suma
pendientes = np.linspace(0, 3, 200)
intercepto = y.mean() - pendientes * x.mean()  # intercepto óptimo dado cada pendiente
scr = [np.sum((y - (b * x + b0))**2) for b, b0 in zip(pendientes, intercepto)]

b_optimo = pendientes[np.argmin(scr)]
b0_optimo = y.mean() - b_optimo * x.mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Izquierda: la recta óptima vs datos ---
axes[0].scatter(x, y, color='tomato', zorder=5, label='Datos')
axes[0].plot(x, b_optimo * x + b0_optimo, 'b-', lw=2,
             label=f'MCO: $\hat{{y}}$ = {b_optimo:.2f}x + {b0_optimo:.2f}')
# Cuadrados de residuos como áreas visuales
for xi, yi in zip(x, y):
    yhat = b_optimo * xi + b0_optimo
    lado = abs(yi - yhat)
    sq = plt.Rectangle((xi - lado/2, min(yi, yhat)), lado, lado,
                        fill=True, alpha=0.15, color='purple')
    axes[0].add_patch(sq)
axes[0].set_title('MCO: minimiza la SUMA de los cuadrados de los residuos\n(cuadrados morados = lo que se minimiza)')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# --- Derecha: función de costo — mínimo en la pendiente óptima ---
axes[1].plot(pendientes, scr, color='steelblue', lw=2)
axes[1].axvline(b_optimo, color='red', ls='--', label=f'Pendiente óptima = {b_optimo:.2f}')
axes[1].scatter([b_optimo], [min(scr)], color='red', s=80, zorder=5)
axes[1].set_title('Suma de Cuadrados de Residuos\nvs. pendiente del modelo')
axes[1].set_xlabel('Pendiente $\\beta_1$'); axes[1].set_ylabel('SCR')
axes[1].legend()

plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. MLE — Máxima Verosimilitud (Maximum Likelihood Estimation)

**Definición:** MLE es un método alternativo para estimar los parámetros de un modelo. En lugar de minimizar errores al cuadrado, busca los parámetros que hacen que los datos observados sean **lo más probables posible**.

**Idea intuitiva:** si lanzas una moneda 10 veces y sale cara 7 veces, ¿cuál es la probabilidad de cara más verosímil? MLE respondería: 7/10 = 0.7, porque ese valor hace que el resultado observado (7 caras en 10 lanzamientos) sea el más probable de todos los posibles.

La **función de verosimilitud** $\mathcal{L}(\theta)$ mide qué tan probable es el conjunto de datos dado un parámetro $\theta$:

$$\mathcal{L}(\theta) = P(\text{datos} \mid \theta) = \prod_{i=1}^n P(y_i \mid x_i, \theta)$$

En la práctica se maximiza el **log de la verosimilitud** (log-likelihood) para evitar underflow numérico al multiplicar probabilidades pequeñas.

| | MCO | MLE |
|---|---|---|
| **Usado en** | Regresión lineal | Regresión logística, modelos probabilísticos |
| **Qué optimiza** | Minimiza $\sum e_i^2$ | Maximiza $P(\text{datos} \mid \theta)$ |
| **Requiere supuesto de normalidad** | Sí (para inferencia) | No necesariamente |
| **Resultado con datos normales** | Son equivalentes | Son equivalentes |

In [ ]:
# Ejemplo: estimar la media de una distribución normal con MLE
# MLE dice que la mejor estimación de la media es simplemente... la media muestral.
# Aquí lo verificamos visualmente.

np.random.seed(42)
datos = np.random.normal(loc=5.0, scale=1.5, size=30)  # datos reales (media verdadera = 5)

# Probamos distintos valores de mu y calculamos el log-likelihood
mu_candidatos = np.linspace(2, 8, 300)
sigma_fija = datos.std()  # asumimos sigma conocida para simplificar

log_likelihoods = [
    np.sum(stats.norm.logpdf(datos, loc=mu, scale=sigma_fija))
    for mu in mu_candidatos
]

mu_mle = mu_candidatos[np.argmax(log_likelihoods)]  # máximo del log-likelihood

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Izquierda: datos y distribución bajo el parámetro MLE ---
xr = np.linspace(0, 10, 200)
axes[0].hist(datos, bins=10, density=True, alpha=0.5, color='steelblue', edgecolor='white', label='Datos observados')
axes[0].plot(xr, stats.norm.pdf(xr, mu_mle, sigma_fija), 'r-', lw=2,
             label=f'Distribución MLE (μ={mu_mle:.2f})')
axes[0].axvline(datos.mean(), color='orange', ls='--', label=f'Media muestral = {datos.mean():.2f}')
axes[0].set_title('MLE estima la μ que mejor\n"explica" los datos observados')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# --- Derecha: curva de log-likelihood ---
axes[1].plot(mu_candidatos, log_likelihoods, color='steelblue', lw=2)
axes[1].axvline(mu_mle, color='red', ls='--', label=f'μ MLE = {mu_mle:.2f}')
axes[1].scatter([mu_mle], [max(log_likelihoods)], color='red', s=80, zorder=5)
axes[1].set_title('Log-Likelihood vs μ\n(buscamos el máximo)')
axes[1].set_xlabel('μ candidato'); axes[1].set_ylabel('Log-Likelihood')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Media muestral: {datos.mean():.4f}  |  μ MLE: {mu_mle:.4f}  → son prácticamente iguales ✓')

---
<a id='4'></a>
## 4. Distribuciones de probabilidad

Una **distribución de probabilidad** describe cómo se reparten las probabilidades sobre los posibles valores de una variable. Son la base de casi todo en ML.

| Distribución | Variable | Parámetros | Dónde aparece en ML |
|---|---|---|---|
| **Normal (Gaussiana)** | Continua | $\mu$, $\sigma$ | Supuesto de MCO, inicialización de pesos en redes |
| **Bernoulli** | Binaria (0/1) | $p$ | Variable objetivo en clasificación binaria |
| **Binomial** | Conteo de éxitos en $n$ intentos | $n$, $p$ | Modelar tasas de conversión |
| **Uniforme** | Continua en $[a,b]$ | $a$, $b$ | Inicialización de pesos, split train/test |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Normal: la campana de Gauss ---
# Dos parámetros: media (centro) y desviación estándar (anchura)
x = np.linspace(-5, 10, 300)
for mu, sigma, color, lbl in [(0, 1, 'steelblue', 'μ=0, σ=1 (estándar)'),
                               (3, 1, 'tomato',    'μ=3, σ=1'),
                               (0, 2, 'green',     'μ=0, σ=2 (más ancha)')]:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), lw=2, color=color, label=lbl)
axes[0].set_title('Distribución Normal\n$f(x) = \\frac{1}{\\sigma\\sqrt{2\\pi}} e^{-(x-\\mu)^2/2\\sigma^2}$')
axes[0].set_xlabel('x'); axes[0].set_ylabel('Densidad')
axes[0].legend(fontsize=7); axes[0].grid(True, alpha=0.3)

# --- Bernoulli: para variable binaria (churn sí/no) ---
# Un solo parámetro p = probabilidad de éxito (y=1)
for p, color, lbl in [(0.3, 'steelblue', 'p=0.3 (poco churn)'),
                      (0.5, 'tomato',    'p=0.5 (balanceado)'),
                      (0.7, 'green',     'p=0.7 (mucho churn)')]:
    axes[1].bar([0 - 0.1 + [0.3,0.5,0.7].index(p)*0.1,
                 1 - 0.1 + [0.3,0.5,0.7].index(p)*0.1],
                [1-p, p], width=0.08, color=color, alpha=0.8, label=lbl)
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['y=0 (No churn)', 'y=1 (Churn)'])
axes[1].set_title('Distribución Bernoulli\n(variable objetivo en clasificación binaria)')
axes[1].set_ylabel('P(y)'); axes[1].legend(fontsize=7); axes[1].grid(True, alpha=0.3, axis='y')

# --- Uniforme: igual probabilidad en todo el rango ---
x_unif = np.linspace(-0.5, 2.5, 300)
axes[2].plot(x_unif, stats.uniform.pdf(x_unif, 0, 2), 'steelblue', lw=2, label='Uniforme [0, 2]')
axes[2].fill_between(x_unif, stats.uniform.pdf(x_unif, 0, 2), alpha=0.2, color='steelblue')
axes[2].set_title('Distribución Uniforme\n(toda zona tiene igual probabilidad)')
axes[2].set_xlabel('x'); axes[2].set_ylabel('Densidad')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. Supuestos estadísticos

Un **supuesto estadístico** es una condición que debe cumplirse para que las conclusiones de un modelo sean válidas. Si se viola, los coeficientes pueden ser correctos pero la **inferencia** (p-valores, intervalos de confianza) se vuelve inválida.

### Supuestos principales de MCO

| Supuesto | Qué significa | Si se viola... |
|----------|--------------|----------------|
| **Linealidad** | La relación entre $X$ e $y$ es lineal | Los residuos tendrán patrón curvo |
| **Normalidad de residuos** | Los $e_i$ siguen una distribución normal | Los p-valores y IC son incorrectos |
| **Homocedasticidad** | La varianza de $e_i$ es constante para todo $x$ | Los IC son demasiado anchos o estrechos |
| **Independencia** | Los $e_i$ no están correlacionados entre sí | Errores estándar subestimados (series temporales) |
| **No multicolinealidad** | Los predictores no están linealmente relacionados | Coeficientes inestables e interpretación imposible |

In [ ]:
np.random.seed(7)
x = np.linspace(1, 10, 50)

# Caso A: homocedasticidad — varianza constante (supuesto cumplido)
residuos_homo = np.random.normal(0, 1.5, len(x))

# Caso B: heterocedasticidad — varianza crece con x (supuesto violado)
residuos_hetero = np.random.normal(0, 0.3 * x, len(x))

# Caso C: no linealidad — residuos con patrón curvo (supuesto de linealidad violado)
residuos_nolineal = np.sin(x) * 2 + np.random.normal(0, 0.3, len(x))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Gráfico de residuos vs x — lo ideal es una nube horizontal sin patrón
configs = [
    (axes[0], residuos_homo,     'steelblue', '✅ Homocedasticidad\n(varianza constante — supuesto cumplido)'),
    (axes[1], residuos_hetero,   'tomato',    '❌ Heterocedasticidad\n(varianza crece con x — supuesto violado)'),
    (axes[2], residuos_nolineal, 'orange',    '❌ No linealidad\n(patrón curvo — supuesto violado)'),
]

for ax, res, color, title in configs:
    ax.scatter(x, res, color=color, alpha=0.7, s=30)
    ax.axhline(0, color='black', lw=1, ls='--')
    ax.set_xlabel('x (valor predicho)'); ax.set_ylabel('Residuo $e_i$')
    ax.set_title(title); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
<a id='6'></a>
## 6. Sesgo y Varianza (Bias-Variance Tradeoff)

El error de un modelo puede descomponerse en tres partes:

$$\text{Error} = \text{Sesgo}^2 + \text{Varianza} + \text{Ruido irreducible}$$

| Componente | Definición | Causa |
|---|---|---|
| **Sesgo (Bias)** | Qué tan lejos están las predicciones del valor real **en promedio** | Modelo demasiado simple (underfitting) |
| **Varianza** | Cuánto cambian las predicciones al entrenar con datos distintos | Modelo demasiado complejo (overfitting) |
| **Ruido** | Error intrínseco de los datos — no se puede eliminar | Datos ruidosos o variable no determinista |

**El dilema:** reducir el sesgo generalmente aumenta la varianza, y viceversa. El objetivo es encontrar el punto de equilibrio.

In [ ]:
np.random.seed(0)
x_true = np.linspace(0, 10, 200)
y_true = np.sin(x_true) + 0.5 * x_true  # relación verdadera (desconocida)

# Simulamos 5 datasets de entrenamiento ligeramente distintos
datasets = [np.sin(x_true) + 0.5 * x_true + np.random.normal(0, 0.8, len(x_true))
            for _ in range(5)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Alto sesgo (modelo lineal — demasiado simple) ---
for i, d in enumerate(datasets):
    m, b = np.polyfit(x_true, d, 1)  # recta (grado 1)
    axes[0].plot(x_true, m * x_true + b, alpha=0.4, color='tomato', lw=1.5,
                 label='Predicción lineal' if i == 0 else '')
axes[0].plot(x_true, y_true, 'k-', lw=2, label='Verdad')
axes[0].set_title('Alto Sesgo / Baja Varianza\n(modelo lineal — underfitting)\nLas líneas son similares entre sí pero alejadas de la verdad')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# --- Alta varianza (polinomio grado 15 — demasiado complejo) ---
for i, d in enumerate(datasets):
    coefs = np.polyfit(x_true, d, 15)  # polinomio de grado 15
    axes[1].plot(x_true, np.polyval(coefs, x_true), alpha=0.4, color='steelblue', lw=1.5,
                 label='Pred. polinomio grado 15' if i == 0 else '')
axes[1].plot(x_true, y_true, 'k-', lw=2, label='Verdad')
axes[1].set_ylim(-3, 8)
axes[1].set_title('Bajo Sesgo / Alta Varianza\n(polinomio grado 15 — overfitting)\nLas curvas varían mucho entre datasets')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
<a id='7'></a>
## 7. Overfitting y Underfitting

| Problema | Qué ocurre | Error en train | Error en test | Solución |
|---|---|---|---|---|
| **Underfitting** | El modelo es demasiado simple para capturar el patrón | Alto | Alto | Modelo más complejo, más features |
| **Buen ajuste** | El modelo generaliza correctamente | Bajo | Bajo | — |
| **Overfitting** | El modelo memoriza el ruido del train | Muy bajo | Alto | Regularización, más datos, modelo más simple |

La señal de alarma del overfitting es una **brecha grande** entre el error de train y el de test.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(3)
x_all = np.sort(np.random.uniform(0, 10, 60))
y_all = np.sin(x_all) + np.random.normal(0, 0.4, len(x_all))

X_all = x_all.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=0)

grados = range(1, 16)
err_train, err_test = [], []

for g in grados:
    poly = PolynomialFeatures(g)
    Xtr_p = poly.fit_transform(X_tr)
    Xte_p = poly.transform(X_te)
    lr = LinearRegression().fit(Xtr_p, y_tr)
    err_train.append(mean_squared_error(y_tr, lr.predict(Xtr_p)))
    err_test.append(mean_squared_error(y_te, lr.predict(Xte_p)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grados, err_train, 'o-', color='steelblue', lw=2, label='Error TRAIN')
ax.plot(grados, err_test,  's-', color='tomato',    lw=2, label='Error TEST')
ax.axvspan(1, 2.5,  alpha=0.08, color='orange', label='Underfitting')
ax.axvspan(2.5, 5,  alpha=0.08, color='green',  label='Zona óptima')
ax.axvspan(5, 15.5, alpha=0.08, color='red',    label='Overfitting')
ax.set_xlabel('Grado del polinomio (complejidad del modelo)')
ax.set_ylabel('MSE')
ax.set_title('Overfitting vs Underfitting\nError Train vs Test según complejidad del modelo')
ax.set_ylim(0, 1.5); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
<a id='8'></a>
## 8. Términos de clasificación: TP, FP, TN, FN

Cuando un modelo de clasificación binaria hace una predicción, hay 4 posibles resultados:

|  | **Predice: Positivo (1)** | **Predice: Negativo (0)** |
|---|---|---|
| **Real: Positivo (1)** | ✅ **TP** — Verdadero Positivo | ❌ **FN** — Falso Negativo *(Error Tipo II)* |
| **Real: Negativo (0)** | ❌ **FP** — Falso Positivo *(Error Tipo I)* | ✅ **TN** — Verdadero Negativo |

**Ejemplo con churn:**
- **TP**: el modelo predice churn → el cliente sí se fue. ✅ Acertó.
- **FP**: el modelo predice churn → el cliente no se fue. ❌ Falsa alarma (se le ofreció descuento innecesario).
- **FN**: el modelo predice no-churn → el cliente sí se fue. ❌ Lo perdimos sin haberlo retenido.
- **TN**: el modelo predice no-churn → el cliente no se fue. ✅ Acertó.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Izquierda: matriz de confusión visual ---
matriz = np.array([[120, 30],   # TN, FP
                   [25,  50]])  # FN, TP

im = axes[0].imshow(matriz, cmap='Blues')
etiquetas = [['TN\n(120)', 'FP\n(30)'], ['FN\n(25)', 'TP\n(50)']]
colores_txt = [['navy', 'navy'], ['navy', 'white']]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, etiquetas[i][j], ha='center', va='center',
                     fontsize=14, fontweight='bold', color=colores_txt[i][j])

axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Pred: No churn (0)', 'Pred: Churn (1)'], fontsize=9)
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['Real: No churn (0)', 'Real: Churn (1)'], fontsize=9)
axes[0].set_title('Matriz de confusión\n(ejemplo con 225 clientes)')

# --- Derecha: métricas derivadas ---
TN, FP, FN, TP = 120, 30, 25, 50
metricas = {
    'Accuracy\n(TP+TN)/total': (TP+TN)/(TP+TN+FP+FN),
    'Precision\nTP/(TP+FP)':   TP/(TP+FP),
    'Recall\nTP/(TP+FN)':      TP/(TP+FN),
    'F1\n2·P·R/(P+R)':         2*TP/(2*TP+FP+FN),
    'Specificity\nTN/(TN+FP)': TN/(TN+FP),
}

nombres = list(metricas.keys())
valores  = list(metricas.values())
colores_bar = ['steelblue', 'tomato', 'green', 'purple', 'orange']

bars = axes[1].barh(nombres, valores, color=colores_bar, alpha=0.8, edgecolor='white')
for bar, val in zip(bars, valores):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlim(0, 1.15)
axes[1].set_title('Métricas derivadas de la\nmatriz de confusión')
axes[1].set_xlabel('Valor (0 = peor, 1 = mejor)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()